<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day04-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 4 — In-class discussion problem (3 of 3)

Work this out **by hand in your group first** — then run the code cell
to check your answer before presenting.

## When does a guide-tree mistake actually hurt?

**Scenario A:** 500 near-identical bacterial genome fragments (all
>95% identical to each other), and you need a rough alignment fast.

**Scenario B:** 5 highly divergent, distantly related protein domains,
and you need the most accurate alignment possible, with time to spare.

**As a group:**

1. Which of the four MSA strategies (exact, progressive, iterative,
   consistency-based) fits each scenario, and why?
2. Progressive alignment's weakness is that an early guide-tree mistake
   propagates forward uncorrected. In which scenario is that mistake
   more dangerous — and why?
3. Does your answer to (2) change which strategy you'd pick for
   Scenario B?

The cell below computes actual pairwise-distance guide trees for a
toy version of each scenario — look at the distance values themselves
as you discuss (2).

In [1]:
import numpy as np
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# Scenario A: 4 near-identical sequences (all pairwise distances small and close together)
D_similar = np.array([
    [0.000, 0.020, 0.021, 0.019],
    [0.020, 0.000, 0.022, 0.018],
    [0.021, 0.022, 0.000, 0.020],
    [0.019, 0.018, 0.020, 0.000],
])

# Scenario B: 4 divergent sequences (all pairwise distances large -- no "safe", close pair)
D_divergent = np.array([
    [0.00, 0.40, 0.45, 0.42],
    [0.40, 0.00, 0.38, 0.44],
    [0.45, 0.38, 0.00, 0.41],
    [0.42, 0.44, 0.41, 0.00],
])

for name, D in [("Scenario A (near-identical)", D_similar), ("Scenario B (divergent)", D_divergent)]:
    Z = linkage(squareform(D, checks=False), method="average")
    print(name)
    print("  pairwise distances range:", D[D > 0].min(), "to", D.max())
    print("  first merge:", Z[0])
    print()

Scenario A (near-identical)
  pairwise distances range: 0.018 to 0.022
  first merge: [1.    3.    0.018 2.   ]

Scenario B (divergent)
  pairwise distances range: 0.38 to 0.45
  first merge: [1.   2.   0.38 2.  ]



**Discussion point:** in Scenario A every pairwise distance sits in a
narrow band (0.018-0.022) — even if the guide tree picks the "wrong"
first pair, both sequences it merges are still >97% identical to
everything else, so the resulting alignment barely changes either way.
In Scenario B every pairwise distance is large (0.38-0.45) — there's no
"safe" wrong choice, because *every* pair being merged is substantially
divergent from every other sequence. A bad early merge there locks a
poor alignment in and, since progressive alignment never revisits it,
that mistake propagates through the rest of the process. This is why
Scenario B is the better fit for a consistency-based method (T-Coffee)
despite being slower: its whole design point is *not* depending on one
early, uncorrectable guide-tree decision.